In [ ]:
````xml
<VSCode.Cell language="markdown">
# Technical Indicator Analysis
## Stock Market Intelligence Dashboard

**Project**: Stock Market Intelligence Dashboard  
**Purpose**: Calculate and analyze technical indicators for stock trading signals  
**Date**: March 2026

This notebook calculates and analyzes key technical indicators:
- Moving Averages (20-day, 50-day)
- Relative Strength Index (RSI)
- MACD (Moving Average Convergence Divergence)
- Bollinger Bands
- Volatility Analysis
- Trading Signals
</VSCode.Cell>

<VSCode.Cell language="code">
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.append('../')

# Import custom modules
from utils.helper_functions import load_data, filter_by_symbol
from utils.financial_metrics import (calculate_rsi, calculate_macd, 
                                     calculate_bollinger_bands, 
                                     get_stock_statistics)
from visualization.charts import plot_moving_averages, plot_volatility

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline

print("✓ Libraries imported successfully")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Load Data

Load the cleaned stock data with calculated features.
</VSCode.Cell>

<VSCode.Cell language="code">
# Load cleaned data
data = load_data('../data/cleaned/stock_data_cleaned.csv')

if data is not None:
    print(f"✓ Data loaded successfully")
    print(f"Shape: {data.shape}")
    print(f"Symbols: {', '.join(data['symbol'].unique())}")
else:
    print("✗ Failed to load data")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Moving Average Analysis

Moving averages smooth out price data and help identify trends.

**20-Day MA**: Short-term trend indicator  
**50-Day MA**: Medium-term trend indicator

**Golden Cross**: When 20-day MA crosses above 50-day MA (bullish signal)  
**Death Cross**: When 20-day MA crosses below 50-day MA (bearish signal)
</VSCode.Cell>

<VSCode.Cell language="code">
# Plot moving averages for each stock
for symbol in data['symbol'].unique():
    print(f"\n{symbol} - Moving Average Analysis")
    print("="*60)
    plot_moving_averages(data, symbol)
</VSCode.Cell>

<VSCode.Cell language="code">
# Identify moving average crossovers
print("="*80)
print("MOVING AVERAGE CROSSOVER ANALYSIS")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol].copy()
    
    # Create signal column
    symbol_data['ma_signal'] = 0
    symbol_data.loc[symbol_data['ma_20'] > symbol_data['ma_50'], 'ma_signal'] = 1  # Bullish
    symbol_data.loc[symbol_data['ma_20'] < symbol_data['ma_50'], 'ma_signal'] = -1  # Bearish
    
    # Find crossovers
    symbol_data['ma_crossover'] = symbol_data['ma_signal'].diff()
    
    golden_crosses = symbol_data[symbol_data['ma_crossover'] == 2]
    death_crosses = symbol_data[symbol_data['ma_crossover'] == -2]
    
    current_signal = "Bullish" if symbol_data['ma_signal'].iloc[-1] == 1 else "Bearish"
    
    print(f"\n{symbol}:")
    print(f"  Current Signal: {current_signal}")
    print(f"  Golden Crosses (last 6 months): {len(golden_crosses[-180:])}")
    print(f"  Death Crosses (last 6 months): {len(death_crosses[-180:])}")
    
    if len(golden_crosses) > 0:
        last_golden = golden_crosses.iloc[-1]
        print(f"  Last Golden Cross: {last_golden['date']}")
    
    if len(death_crosses) > 0:
        last_death = death_crosses.iloc[-1]
        print(f"  Last Death Cross: {last_death['date']}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Relative Strength Index (RSI)

RSI measures the speed and magnitude of price changes.

**Interpretation**:
- RSI > 70: Overbought (potential sell signal)
- RSI < 30: Oversold (potential buy signal)
- RSI 30-70: Neutral zone
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate RSI for each stock
for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol].copy()
    
    # Calculate RSI
    symbol_data['rsi'] = calculate_rsi(symbol_data['close'], window=14)
    
    # Update main dataframe
    data.loc[data['symbol'] == symbol, 'rsi'] = symbol_data['rsi'].values

print("✓ RSI calculated for all stocks")
</VSCode.Cell>

<VSCode.Cell language="code">
# Visualize RSI
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

symbols = data['symbol'].unique()

for idx, symbol in enumerate(symbols):
    symbol_data = data[data['symbol'] == symbol]
    
    ax = axes[idx]
    ax.plot(symbol_data['date'], symbol_data['rsi'], linewidth=2, color='#2E86AB')
    ax.axhline(y=70, color='red', linestyle='--', linewidth=1.5, label='Overbought (70)')
    ax.axhline(y=30, color='green', linestyle='--', linewidth=1.5, label='Oversold (30)')
    ax.fill_between(symbol_data['date'], 30, 70, alpha=0.1, color='gray')
    ax.set_title(f'{symbol} - RSI (14-day)', fontweight='bold')
    ax.set_ylabel('RSI')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../data/processed/rsi_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="code">
# RSI-based signals
print("="*80)
print("CURRENT RSI SIGNALS")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol]
    current_rsi = symbol_data['rsi'].iloc[-1]
    
    if current_rsi > 70:
        signal = "OVERBOUGHT - Consider Selling"
        color = "🔴"
    elif current_rsi < 30:
        signal = "OVERSOLD - Consider Buying"
        color = "🟢"
    else:
        signal = "NEUTRAL"
        color = "⚪"
    
    print(f"{color} {symbol}: RSI = {current_rsi:.2f} - {signal}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. MACD Analysis

MACD shows the relationship between two moving averages of a stock's price.

**Components**:
- MACD Line: 12-day EMA - 26-day EMA
- Signal Line: 9-day EMA of MACD
- Histogram: MACD - Signal

**Signals**:
- MACD crosses above Signal: Bullish
- MACD crosses below Signal: Bearish
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate MACD for each stock
for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol].copy()
    
    # Calculate MACD
    macd_data = calculate_macd(symbol_data['close'])
    
    symbol_data['macd'] = macd_data['macd'].values
    symbol_data['macd_signal'] = macd_data['signal'].values
    symbol_data['macd_histogram'] = macd_data['histogram'].values
    
    # Update main dataframe
    data.loc[data['symbol'] == symbol, 'macd'] = symbol_data['macd'].values
    data.loc[data['symbol'] == symbol, 'macd_signal'] = symbol_data['macd_signal'].values
    data.loc[data['symbol'] == symbol, 'macd_histogram'] = symbol_data['macd_histogram'].values

print("✓ MACD calculated for all stocks")
</VSCode.Cell>

<VSCode.Cell language="code">
# Visualize MACD
fig, axes = plt.subplots(4, 2, figsize=(16, 14))

symbols = data['symbol'].unique()

for idx, symbol in enumerate(symbols):
    symbol_data = data[data['symbol'] == symbol]
    
    # Price chart
    ax1 = axes[idx, 0]
    ax1.plot(symbol_data['date'], symbol_data['close'], linewidth=2, color='#2E86AB')
    ax1.set_title(f'{symbol} - Close Price', fontweight='bold')
    ax1.set_ylabel('Price ($)')
    ax1.grid(True, alpha=0.3)
    
    # MACD chart
    ax2 = axes[idx, 1]
    ax2.plot(symbol_data['date'], symbol_data['macd'], linewidth=2, 
            color='#2E86AB', label='MACD')
    ax2.plot(symbol_data['date'], symbol_data['macd_signal'], linewidth=2, 
            color='#F18F01', label='Signal')
    ax2.bar(symbol_data['date'], symbol_data['macd_histogram'], 
           color='gray', alpha=0.3, label='Histogram')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_title(f'{symbol} - MACD', fontweight='bold')
    ax2.set_ylabel('MACD Value')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Date')
axes[-1, 1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../data/processed/macd_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="code">
# MACD signals
print("="*80)
print("CURRENT MACD SIGNALS")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol]
    
    current_macd = symbol_data['macd'].iloc[-1]
    current_signal = symbol_data['macd_signal'].iloc[-1]
    current_histogram = symbol_data['macd_histogram'].iloc[-1]
    
    if current_macd > current_signal:
        signal = "BULLISH - Buy Signal"
        color = "🟢"
    else:
        signal = "BEARISH - Sell Signal"
        color = "🔴"
    
    print(f"{color} {symbol}:")
    print(f"     MACD: {current_macd:.4f}")
    print(f"     Signal: {current_signal:.4f}")
    print(f"     Histogram: {current_histogram:.4f}")
    print(f"     Signal: {signal}\n")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Bollinger Bands Analysis

Bollinger Bands measure volatility and provide relative price levels.

**Bands**:
- Upper Band: 20-day MA + (2 × std dev)
- Middle Band: 20-day MA
- Lower Band: 20-day MA - (2 × std dev)

**Interpretation**:
- Price near upper band: Overbought
- Price near lower band: Oversold
- Bands widening: High volatility
- Bands narrowing: Low volatility
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate Bollinger Bands for each stock
for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol].copy()
    
    # Calculate Bollinger Bands
    bb_data = calculate_bollinger_bands(symbol_data['close'])
    
    symbol_data['bb_upper'] = bb_data['upper'].values
    symbol_data['bb_middle'] = bb_data['middle'].values
    symbol_data['bb_lower'] = bb_data['lower'].values
    
    # Update main dataframe
    data.loc[data['symbol'] == symbol, 'bb_upper'] = symbol_data['bb_upper'].values
    data.loc[data['symbol'] == symbol, 'bb_middle'] = symbol_data['bb_middle'].values
    data.loc[data['symbol'] == symbol, 'bb_lower'] = symbol_data['bb_lower'].values

print("✓ Bollinger Bands calculated for all stocks")
</VSCode.Cell>

<VSCode.Cell language="code">
# Visualize Bollinger Bands
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

symbols = data['symbol'].unique()

for idx, symbol in enumerate(symbols):
    symbol_data = data[data['symbol'] == symbol]
    
    ax = axes[idx]
    ax.plot(symbol_data['date'], symbol_data['close'], linewidth=2, 
           color='black', label='Close Price')
    ax.plot(symbol_data['date'], symbol_data['bb_upper'], linewidth=1.5, 
           linestyle='--', color='red', label='Upper Band')
    ax.plot(symbol_data['date'], symbol_data['bb_middle'], linewidth=1.5, 
           linestyle='--', color='blue', label='Middle Band (MA)')
    ax.plot(symbol_data['date'], symbol_data['bb_lower'], linewidth=1.5, 
           linestyle='--', color='green', label='Lower Band')
    ax.fill_between(symbol_data['date'], symbol_data['bb_lower'], 
                    symbol_data['bb_upper'], alpha=0.1, color='gray')
    ax.set_title(f'{symbol} - Bollinger Bands', fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
axes[-2].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../data/processed/bollinger_bands.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="code">
# Bollinger Band position analysis
print("="*80)
print("CURRENT BOLLINGER BAND POSITION")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol]
    
    current_price = symbol_data['close'].iloc[-1]
    upper_band = symbol_data['bb_upper'].iloc[-1]
    middle_band = symbol_data['bb_middle'].iloc[-1]
    lower_band = symbol_data['bb_lower'].iloc[-1]
    
    band_width = upper_band - lower_band
    position = ((current_price - lower_band) / band_width) * 100
    
    if position > 80:
        signal = "Near Upper Band - Potentially Overbought"
        color = "🔴"
    elif position < 20:
        signal = "Near Lower Band - Potentially Oversold"
        color = "🟢"
    else:
        signal = "Within Normal Range"
        color = "⚪"
    
    print(f"{color} {symbol}:")
    print(f"     Current Price: ${current_price:.2f}")
    print(f"     Upper Band: ${upper_band:.2f}")
    print(f"     Middle Band: ${middle_band:.2f}")
    print(f"     Lower Band: ${lower_band:.2f}")
    print(f"     Position: {position:.1f}% of band width")
    print(f"     Signal: {signal}\n")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6. Volatility Analysis

Analyze rolling volatility to identify periods of high and low market uncertainty.
</VSCode.Cell>

<VSCode.Cell language="code">
# Plot volatility comparison
plot_volatility(data, save_path='../data/processed/volatility_comparison.png')
</VSCode.Cell>

<VSCode.Cell language="code">
# Volatility statistics
print("="*80)
print("VOLATILITY STATISTICS")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol]
    
    current_vol = symbol_data['volatility'].iloc[-1]
    avg_vol = symbol_data['volatility'].mean()
    max_vol = symbol_data['volatility'].max()
    min_vol = symbol_data['volatility'].min()
    
    print(f"\n{symbol}:")
    print(f"  Current Volatility: {current_vol:.2f}%")
    print(f"  Average Volatility: {avg_vol:.2f}%")
    print(f"  Max Volatility: {max_vol:.2f}%")
    print(f"  Min Volatility: {min_vol:.2f}%")
    
    if current_vol > avg_vol * 1.5:
        print(f"  ⚠️ HIGH VOLATILITY - Current volatility is {current_vol/avg_vol:.1f}x average")
    elif current_vol < avg_vol * 0.5:
        print(f"  ℹ️ LOW VOLATILITY - Current volatility is {current_vol/avg_vol:.1f}x average")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7. Comprehensive Stock Statistics

Calculate all key metrics for each stock.
</VSCode.Cell>

<VSCode.Cell language="code">
# Get comprehensive statistics
print("="*80)
print("COMPREHENSIVE STOCK STATISTICS")
print("="*80)

all_stats = []

for symbol in data['symbol'].unique():
    stats = get_stock_statistics(data, symbol)
    all_stats.append(stats)
    
    print(f"\n{symbol}:")
    print(f"  Period: {stats['start_date']} to {stats['end_date']}")
    print(f"  Price Range: ${stats['min_price']:.2f} - ${stats['max_price']:.2f}")
    print(f"  Total Return: {stats['total_return']:.2f}%")
    print(f"  Avg Daily Return: {stats['avg_daily_return']:.4f}%")
    print(f"  Volatility (Annualized): {stats['volatility']:.2f}%")
    print(f"  Max Drawdown: {stats['max_drawdown']:.2f}%")
    print(f"  Sharpe Ratio: {stats['sharpe_ratio']:.2f}")
    print(f"  Avg Volume: {stats['avg_volume']:,.0f}")

# Create DataFrame
stats_df = pd.DataFrame(all_stats)
stats_df.to_csv('../data/processed/stock_statistics.csv', index=False)
print("\n✓ Statistics exported to: ../data/processed/stock_statistics.csv")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 8. Trading Signal Summary

Combine all indicators to generate trading recommendations.
</VSCode.Cell>

<VSCode.Cell language="code">
# Generate trading signals summary
print("="*80)
print("TRADING SIGNAL SUMMARY")
print("="*80)

for symbol in data['symbol'].unique():
    symbol_data = data[data['symbol'] == symbol]
    
    # Get latest values
    ma_20 = symbol_data['ma_20'].iloc[-1]
    ma_50 = symbol_data['ma_50'].iloc[-1]
    rsi = symbol_data['rsi'].iloc[-1]
    macd = symbol_data['macd'].iloc[-1]
    macd_signal = symbol_data['macd_signal'].iloc[-1]
    current_price = symbol_data['close'].iloc[-1]
    bb_upper = symbol_data['bb_upper'].iloc[-1]
    bb_lower = symbol_data['bb_lower'].iloc[-1]
    
    # Count signals
    buy_signals = 0
    sell_signals = 0
    
    # MA Signal
    if ma_20 > ma_50:
        buy_signals += 1
        ma_signal = "Bullish"
    else:
        sell_signals += 1
        ma_signal = "Bearish"
    
    # RSI Signal
    if rsi < 30:
        buy_signals += 1
        rsi_signal = "Oversold (Buy)"
    elif rsi > 70:
        sell_signals += 1
        rsi_signal = "Overbought (Sell)"
    else:
        rsi_signal = "Neutral"
    
    # MACD Signal
    if macd > macd_signal:
        buy_signals += 1
        macd_sig = "Bullish"
    else:
        sell_signals += 1
        macd_sig = "Bearish"
    
    # Bollinger Band Signal
    if current_price < bb_lower:
        buy_signals += 1
        bb_signal = "Below Lower Band (Buy)"
    elif current_price > bb_upper:
        sell_signals += 1
        bb_signal = "Above Upper Band (Sell)"
    else:
        bb_signal = "Within Bands"
    
    # Overall recommendation
    total_signals = buy_signals + sell_signals
    if buy_signals > sell_signals:
        recommendation = "🟢 BUY"
    elif sell_signals > buy_signals:
        recommendation = "🔴 SELL"
    else:
        recommendation = "⚪ HOLD"
    
    print(f"\n{symbol} - {recommendation}")
    print(f"  Moving Averages: {ma_signal}")
    print(f"  RSI: {rsi_signal} ({rsi:.2f})")
    print(f"  MACD: {macd_sig}")
    print(f"  Bollinger Bands: {bb_signal}")
    print(f"  Signal Score: {buy_signals} Buy / {sell_signals} Sell")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 9. Export Enhanced Data

Save the data with all calculated technical indicators.
</VSCode.Cell>

<VSCode.Cell language="code">
# Export data with technical indicators
data.to_csv('../data/processed/stock_data_with_indicators.csv', index=False)
print("✓ Data with technical indicators exported to:")
print("  ../data/processed/stock_data_with_indicators.csv")
print(f"\nTotal columns: {len(data.columns)}")
print(f"New indicator columns: rsi, macd, macd_signal, macd_histogram,")
print(f"                      bb_upper, bb_middle, bb_lower")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Conclusion

This technical analysis revealed:

### Key Findings:
1. **Moving Averages**: Identified trend directions and crossover signals
2. **RSI**: Identified overbought/oversold conditions
3. **MACD**: Confirmed momentum and trend strength
4. **Bollinger Bands**: Measured volatility and price extremes
5. **Volatility**: Tracked risk levels over time

### Trading Insights:
- Combined indicators provide more reliable signals than single indicators
- High volatility periods require different strategies than low volatility periods
- Crossovers and divergences can signal potential trend reversals

**Next Steps**:
- Portfolio construction and optimization
- Backtesting trading strategies
- Risk management analysis

---
**End of Technical Indicator Analysis**
</VSCode.Cell>
````